In [6]:
import pyhmmer
from pathlib import Path
from Bio import SeqIO

seq_file = "final.contigs.fa"
hmm_files = list(Path(".").glob("*.hmm"))

# Read DNA sequences and translate to protein using Biopython
protein_sequences = []
for record in SeqIO.parse(seq_file, "fasta"):
    # Translate DNA to protein (forward frame 1)
    protein_seq = str(record.seq.translate())
    
    # Create pyhmmer TextSequence then digitize
    text_seq = pyhmmer.easel.TextSequence(
        name=record.id.encode(), 
        sequence=protein_seq.encode()
    )
    protein_sequences.append(text_seq.digitize(pyhmmer.easel.Alphabet.amino()))

print(f"Translated {len(protein_sequences)} sequences to protein")

# Run hmmsearch
for hmm_file in hmm_files:
    print(f"\nSearching with {hmm_file.name}...")
    
    with pyhmmer.plan7.HMMFile(hmm_file) as hmm_handle:
        hmms = list(hmm_handle)
    
    hits = pyhmmer.hmmer.hmmsearch(hmms, protein_sequences, cpus=4)
    total_hits = sum(len(hit) for hit in hits)
    print(f"Found {total_hits} hits")

d:\Users\User\miniconda3\envs\virus\Lib\site-packages\Bio\Seq.py:2877: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


Translated 37500 sequences to protein

Searching with 1A.hmm...
Found 14 hits

Searching with 1B.hmm...
Found 10 hits

Searching with 2.hmm...
Found 6 hits

Searching with astro_1b_pcr_primer_region_RDRP.hmm...
Found 14 hits

Searching with kobuvirus.hmm...
Found 42 hits

Searching with kobu_P1.hmm...
Found 17 hits

Searching with kobu_P2.hmm...
Found 13 hits

Searching with kobu_P3.hmm...
Found 22 hits

Searching with parechovirus.hmm...
Found 40 hits

Searching with parecho_P1.hmm...
Found 13 hits

Searching with parecho_P2.hmm...
Found 13 hits

Searching with parecho_P3.hmm...
Found 22 hits


In [8]:
import pyhmmer
from pathlib import Path
from Bio import SeqIO
import pandas as pd

seq_file = "final.contigs.fa"
hmm_files = list(Path(".").glob("*.hmm"))

# Store original DNA sequences for extraction
dna_records = {record.id: str(record.seq) for record in SeqIO.parse(seq_file, "fasta")}

# Translate to protein
protein_sequences = []
sequence_ids = []

for record in SeqIO.parse(seq_file, "fasta"):
    protein_seq = str(record.seq.translate())
    text_seq = pyhmmer.easel.TextSequence(
        name=record.id.encode(), 
        sequence=protein_seq.encode()
    )
    protein_sequences.append(text_seq.digitize(pyhmmer.easel.Alphabet.amino()))
    sequence_ids.append(record.id)

print(f"Translated {len(protein_sequences)} sequences to protein")

# Store results
all_hits = []

for hmm_file in hmm_files:
    print(f"\nSearching with {hmm_file.name}...")
    
    with pyhmmer.plan7.HMMFile(hmm_file) as hmm_handle:
        hmms = list(hmm_handle)
    
    hits = pyhmmer.hmmer.hmmsearch(hmms, protein_sequences, cpus=4)
    
    for hit_index, hit in enumerate(hits):
        for domain in hit.domains:
            for result in domain:
                target_id = sequence_ids[hit_index]
                all_hits.append({
                    'query_hmm': hmm_file.name.replace('.hmm', ''),
                    'target_contig': target_id,
                    'protein_score': result.score,
                    'evalue': result.domain_ievalue,
                    'protein_start': result.alignment.target_start,
                    'protein_end': result.alignment.target_end,
                    'protein_sequence': result.alignment.target_sequence.decode() if hasattr(result.alignment, 'target_sequence') else ''
                })
    
    print(f"  Found {sum(len(hit) for hit in hits)} hits")

# Create and save table
if all_hits:
    df = pd.DataFrame(all_hits)
    df = df.sort_values('evalue')
    df.to_csv('hmmsearch_detailed.csv', index=False)
    
    print(f"\n{'='*60}")
    print(f"Total hits: {len(df)}")
    print(f"{'='*60}")
    print(df.to_string())
else:
    print("\nNo hits found")

Translated 37500 sequences to protein

Searching with 1A.hmm...


AttributeError: 'pyhmmer.plan7.TopHits' object has no attribute 'domains'